# Shrink Florence-2 with GPTQ — `nn.Linear` only

**Part 2 of 5** · [Open in Colab](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant_A.ipynb)

![Pipeline](images/gptq_pipeline_steps.png)

## Pipeline (run top → bottom)

```
Setup → Load → Scan → Hook → Capture → Rank → Plan → Quantize → Pack → Swap → Verify → Scorecard
```

| Step | Class | Why |
|------|-------|-----|
| 0 | config, `ensure_numpy_stack` | Pin deps; fix Colab import errors |
| 1 | `FlorenceModelLoader`, `ImagePadder`, `FlorenceOCRDetector` | Load model + calibration image |
| 2 | `LinearLayerScanner`, `LinearInventory` | Find `nn.Linear` layers (only ones we quantize) |
| 3–4 | `CalibrationSession` | **Hook** + **capture** activations $X_l$ from real OCR |
| 5 | `SensitivityAnalyzer` | Rank fragile layers (output MSE, not weight MSE) |
| 6 | `QuantPlanBuilder` | Assign int4 / int8 / fp16 per layer |
| 7–9 | `GPTQQuantizer`, `GPTQApplier`, `ModelPatcher` | GPTQ quantize → `GPTQLinear` → swap in model |
| 10–11 | `FlorenceOCRDetector` | OCR verify + scorecard vs fp16 & naive baseline |

**Phases:** A = naive int4 floor · B = scan · C = hook+rank · D = GPTQ plan+apply

**Scope:** quantize only `nn.Linear` — LayerNorm / embeddings stay fp16.

**Run:** GPU required · `MIXED_PRECISION=True` · don't skip Phase A · restart + Run all if pip updates deps.


## Step 0 — Install & config

Pins `numpy==2.1.3`, `scipy==1.14.1`, `scikit-learn==1.6.1`, `transformers==4.49.0`.

| Key setting | Default | Why |
|-------------|---------|-----|
| `MIXED_PRECISION` | True | int4/int8/fp16 per layer |
| `FP16_SENSITIVE_PCT` | 15 | Top fragile layers → fp16 |
| `INT8_MID_PCT` | 35 | Next tier → int8 GPTQ |
| `MAX_CALIB_BATCHES` | 8 | OCR passes for hook calibration |


In [ ]:
import os, re, subprocess, sys

MODEL_ID = "microsoft/Florence-2-base-ft"
OCR_PROMPT = "<OCR_WITH_REGION>"

MIXED_PRECISION = True
BITS = 4
MAX_CALIB_BATCHES = 8
MAX_QUANT_LAYERS = None
MAX_ANALYZE_LAYERS = None
FP16_SENSITIVE_PCT = 15
INT8_MID_PCT = 35
MIN_PARAMS_TO_QUANT = 4096
ALWAYS_FP16_PATTERNS = ("lm_head", "embed", "vision", "patch_embed")
GPTQ_BLOCK_SIZE = 128
GPTQ_DAMPING = 0.01

def _pip_version(pkg):
    # Why: read installed version so we only reinstall when needed
    r = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                       capture_output=True, text=True, check=False)
    m = re.search(r"^Version: (.+)$", r.stdout, re.M)
    return m.group(1) if m else ""

def ensure_numpy_stack():
    # Why: Colab Py3.13 ships mismatched numpy/scipy/sklearn → ImportError: _slice
    # transformers → sklearn → scipy → numpy — all three must match
    want = {"numpy": "2.1.3", "scipy": "1.14.1", "scikit-learn": "1.6.1"}
    if any(_pip_version(p) != v for p, v in want.items()):
        pkgs = [f"{p}=={v}" for p, v in want.items()]
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", *pkgs])
    import numpy as np
    import scipy
    import sklearn
    print(f"numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")

def ensure_transformers():
    # Why: Florence-2 breaks on transformers > 4.49 — pin the version here
    ensure_numpy_stack()
    ok = lambda v: v.startswith("4.49")
    ver = _pip_version("transformers")
    if not ok(ver):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
            "transformers==4.49.0"])
        ver = _pip_version("transformers")
    import transformers
    if not ok(transformers.__version__):
        print("Restart runtime, then Run all."); os.kill(os.getpid(), 9)
    return ver

# Why: install deps in safe order — numpy stack first, then torch + transformers
ensure_numpy_stack()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "accelerate", "pillow", "matplotlib", "requests", "huggingface_hub"])
print(f"Ready — transformers {ensure_transformers()}")


## Step 1 — Core library

| Cell | Classes | Role |
|------|---------|------|
| 1b | `SymmetricQuantizer`, `GPTQState`, `RTNState` | Pack/unpack int4 |
| 1c | `GPTQLinear`, `RTNLinear` | Quantized `nn.Linear` modules |
| 1d | `RTNQuantizer`, `GPTQQuantizer` | Naive RTN vs Hessian GPTQ |
| 1e | `QuantLinearFactory`, `LinearMetrics`, `LinearLayerScanner`, `ModelPatcher` | Build, measure, scan, swap |
| 1f | `CalibrationSession` | Hook + capture (see below) |
| 1g | `QuantMethodComparator` | RTN vs GPTQ compare (Step 15) |

Run cells **1a → 1g** in order.


In [ ]:
# 1a — Imports
from __future__ import annotations
import math, time
from abc import ABC
from dataclasses import dataclass
from io import BytesIO

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUANT_MODULE_TYPES = (nn.Linear,)
print(f"Device: {DEVICE}")


In [ ]:
class SymmetricQuantizer:
    # Why: atomic pack/unpack — every quantizer (RTN, GPTQ) uses this
    def __init__(self, n_bits: int = 4):
        self.n_bits = n_bits

    def qmax(self) -> int:
        # Why: max int value for n_bits (e.g. 7 for int4 symmetric)
        return 2 ** (self.n_bits - 1) - 1

    def quantize(self, W: torch.Tensor):
        # Why: round fp16 weights to int4, keep per-row scale for dequant
        qmax = self.qmax()
        scales = W.abs().amax(1).clamp(min=1e-8) / qmax
        q = torch.round(W / scales.unsqueeze(1)).clamp(-qmax - 1, qmax).to(torch.int8)
        return q, scales

    def dequantize(self, q, scales):
        # Why: reconstruct fp16 weights from packed int4 for forward pass
        return q.float() * scales.unsqueeze(1)


@dataclass
class GPTQState:
    # Why: holds GPTQ-packed weights ready to load into GPTQLinear
    weight_q: torch.Tensor
    weight_scales: torch.Tensor
    bias: torch.Tensor | None
    n_bits: int = 4
    method: str = "gptq"

@dataclass
class RTNState:
    # Why: holds naive round-to-nearest packed weights for RTNLinear
    weight_q: torch.Tensor
    weight_scales: torch.Tensor
    bias: torch.Tensor | None
    n_bits: int = 4
    method: str = "rtn"

QuantState = GPTQState | RTNState
print("SymmetricQuantizer + states OK")

In [ ]:
class BaseQuantLinear(nn.Module, ABC):
    # Why: drop-in replacement for nn.Linear — swap without changing the graph
    def __init__(self, in_f: int, out_f: int, state: QuantState):
        super().__init__()
        self.in_features, self.out_features = in_f, out_f
        self.n_bits, self.method = state.n_bits, state.method
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None
        self._sq = SymmetricQuantizer(state.n_bits)

    @property
    def weight_fp(self):
        # Why: dequant on-the-fly so forward() works like a normal Linear
        return self._sq.dequantize(self.weight_q, self.weight_scales)

    def forward(self, x):
        # Why: same API as nn.Linear — rest of model needs no changes
        return F.linear(x, self.weight_fp.to(x.dtype), self.bias)

    def storage_bytes(self):
        # Why: measure compression ratio vs fp16 (2 bytes/param)
        n = self.weight_q.numel() + self.weight_scales.numel()
        return (n + (self.bias.numel() if self.bias is not None else 0)) * 4

class GPTQLinear(BaseQuantLinear):
    # Why: holds GPTQ-quantized weights (Phase D)
    def __init__(self, in_f, out_f, state: GPTQState):
        super().__init__(in_f, out_f, state)

class RTNLinear(BaseQuantLinear):
    # Why: holds naive RTN weights (Phase A baseline)
    def __init__(self, layer: nn.Linear, state: RTNState):
        super().__init__(layer.in_features, layer.out_features, state)

print("GPTQLinear / RTNLinear OK")

In [ ]:
class RTNQuantizer:
    # Why: naive round-to-nearest — fast baseline for Phase A
    def __init__(self, layer: nn.Linear, n_bits: int = 4):
        self.layer, self.n_bits = layer, n_bits
        self._sq = SymmetricQuantizer(n_bits)

    def quantize(self) -> RTNState:
        # Why: just round weights — no calibration data needed
        q, s = self._sq.quantize(self.layer.weight.data.float())
        b = self.layer.bias.data.clone() if self.layer.bias is not None else None
        return RTNState(q, s, b, self.n_bits)

class GPTQQuantizer:
    # Why: Hessian-aware quant — minimizes output error, not just weight error
    def __init__(self, layer: nn.Linear, n_bits=4, block_size=128, damping=0.01):
        self.layer = layer
        self.n_bits, self.block_size, self.damping = n_bits, block_size, damping
        self.H, self.nsamples = None, 0
        self._sq = SymmetricQuantizer(n_bits)

    def add_batch(self, inp: torch.Tensor):
        # Why: accumulate input covariance (Hessian) from OCR calibration forwards
        if inp.dim() == 3: inp = inp.reshape(-1, inp.shape[-1])
        inp = inp.float()
        if self.H is None:
            self.H = torch.zeros(inp.shape[1], inp.shape[1], device=inp.device)
        self.H += inp.t() @ inp
        self.nsamples += inp.shape[0]

    def quantize(self) -> GPTQState:
        # Why: quantize column-by-column, propagate error to remaining columns
        W = self.layer.weight.data.float().clone()
        H = self.H.clone()
        dead = torch.diag(H) == 0
        H[dead, dead] = 1.0; W[:, dead] = 0.0
        H[torch.arange(H.shape[0], device=H.device), torch.arange(H.shape[0], device=H.device)] += self.damping * H.diag().mean()
        H = torch.linalg.cholesky(H)
        Hinv = torch.linalg.cholesky(torch.linalg.cholesky_inverse(H), upper=True)
        Q, qmax = torch.zeros_like(W), self._sq.qmax()
        for i1 in range(0, W.shape[1], self.block_size):
            i2 = min(i1 + self.block_size, W.shape[1])
            W1, Q1, Err1 = W[:, i1:i2].clone(), torch.zeros_like(W[:, i1:i2]), torch.zeros_like(W[:, i1:i2])
            Hinv1 = Hinv[i1:i2, i1:i2]
            for i in range(i2 - i1):
                w, d = W1[:, i], Hinv1[i, i]
                sc = w.abs().max().clamp(min=1e-8) / qmax
                q = torch.round(w / sc).clamp(-qmax - 1, qmax)
                Q1[:, i] = q
                err = (w - q * sc) / d
                W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)
                Err1[:, i] = err
            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]
        wq, ws = self._sq.quantize(Q)
        return GPTQState(wq.cpu(), ws.cpu(),
            self.layer.bias.detach().cpu() if self.layer.bias is not None else None, self.n_bits)

print("RTNQuantizer + GPTQQuantizer OK")

In [ ]:
class QuantLinearFactory:
    # Why: one place to build the right quant layer from a state object
    @staticmethod
    def build(layer: nn.Linear, state: QuantState) -> BaseQuantLinear:
        if isinstance(state, GPTQState):
            return GPTQLinear(layer.in_features, layer.out_features, state)
        return RTNLinear(layer, state)

class LinearMetrics:
    # Why: shared error metrics used in every phase
    @staticmethod
    def num_params(layer: nn.Linear) -> int:
        # Why: count weight + bias params for inventory and compression ratio
        return layer.weight.numel() + (layer.bias.numel() if layer.bias is not None else 0)

    @staticmethod
    def weight_mse(orig: nn.Linear, quant: BaseQuantLinear) -> float:
        # Why: how far packed weights drift from fp16 (cheap but incomplete metric)
        return (orig.weight.float() - quant.weight_fp).pow(2).mean().item()

    @staticmethod
    def output_mse(orig, quant, inputs) -> float:
        # Why: measures real impact on layer output using OCR calibration activations
        if inputs is None or inputs.numel() == 0: return float("nan")
        x = inputs[:2048].to(orig.weight.device)
        with torch.no_grad():
            y0 = F.linear(x, orig.weight.float(), orig.bias)
            y1 = quant(x)
        return (y0 - y1).pow(2).mean().item()

class LinearLayerScanner:
    # Why: find every nn.Linear — we only quantize Linear layers
    def __init__(self, root: nn.Module):
        self.root = root

    def layers(self, limit=None):
        # Why: walk model tree and return (name, module) pairs
        out = [(n, m) for n, m in self.root.named_modules() if isinstance(m, nn.Linear)]
        return out[:limit] if limit else out

    def is_protected(self, name: str) -> bool:
        # Why: lm_head / embed layers must stay fp16 — quantizing breaks text output
        n = name.lower()
        return any(p in n for p in ALWAYS_FP16_PATTERNS)

class ModelPatcher:
    # Why: surgically swap one layer by name without rebuilding the whole model
    @staticmethod
    def replace(model, name, mod):
        parent_name, _, child_name = name.rpartition(".")
        parent = model.get_submodule(parent_name) if parent_name else model
        setattr(parent, child_name, mod)

print("Factory + metrics + scanner + patcher OK")

#### 1f — `CalibrationSession` · Hook → Capture

![Why hooks](images/gptq_hooks_why_required.png)

**Rule:** need activation stats from real data → **hooks required**. Weight-only quant → **no hooks**.

| Method | Hooks? | Why |
|--------|--------|-----|
| **RTN** (Phase A) | No | Rounds weights only |
| **GPTQ** (Phase D) | Yes | Builds Hessian $H_l=X_l^TX_l$ via `add_batch()` |
| **AWQ / SmoothQuant** | Yes | Need activation scales (notebook 02 full) |
| **Sensitivity** (Phase C) | Yes | `collect_inputs()` saves $X_l$ for output MSE |

**Flow:** register hook → `run(OCR)` → hook saves $X_l$ → `h.remove()`

| Method | Function | Phase |
|--------|----------|-------|
| `collect_inputs()` | save $X_l$ → `captures` | C |
| `register_gptq_hooks()` | stream $X_l$ → `add_batch()` | D |


In [ ]:
class CalibrationSession:
    # Why: GPTQ needs real OCR activations — not random Gaussian inputs
    def __init__(self, model, processor, image, prompt=OCR_PROMPT, device=DEVICE):
        self.model, self.processor, self.image = model, processor, image
        self.prompt, self.device = prompt, device

    def _padded(self):
        # Why: Florence-2 expects square images — pad with white borders
        w, h = self.image.size
        side = max(w, h)
        canvas = Image.new("RGB", (side, side), "white")
        px, py = (side - w) // 2, (side - h) // 2
        canvas.paste(self.image, (px, py))
        return canvas

    def run(self, n_batches: int):
        # Why: run OCR forwards — hooks (if registered) fire here and capture X_l
        padded = self._padded()
        for _ in range(n_batches):
            inp = self.processor(text=self.prompt, images=padded, return_tensors="pt").to(self.device)
            inp["pixel_values"] = inp["pixel_values"].to(dtype=next(self.model.parameters()).dtype)
            with torch.no_grad():
                self.model.generate(input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
                                    max_new_tokens=64, do_sample=False, num_beams=1)

    def register_gptq_hooks(self, quantizers: dict):
        # Why add hook: PyTorch does not expose layer inputs — hook is required to
        # stream X_l into GPTQQuantizer.add_batch() and build Hessian H_l = X_l^T X_l
        handles = []
        for name, q in quantizers.items():
            def hook(m, inp, out, quantizer=q):
                x = inp[0] if isinstance(inp, tuple) else inp
                if x is not None: quantizer.add_batch(x.detach())
            handles.append(self.model.get_submodule(name).register_forward_hook(hook))
        return handles  # Why: keep handles so we can h.remove() after calibration

    def collect_inputs(self, layer_names, n_batches):
        # Why add hook: we need saved X_l per layer for SensitivityAnalyzer (Phase C)
        # Without hook we cannot measure output MSE on real OCR activations
        store = {n: [] for n in layer_names}
        def make_hook(n):
            def hook(m, inp, out):
                x = inp[0] if isinstance(inp, tuple) else inp
                if x is not None: store[n].append(x.detach().reshape(-1, x.shape[-1]).cpu())
            return hook
        handles = [self.model.get_submodule(n).register_forward_hook(make_hook(n)) for n in layer_names]
        self.run(n_batches)  # hooks fire during these OCR forwards
        for h in handles: h.remove()  # Why remove: hooks must not stay on during inference
        return {n: torch.cat(v, 0) if v else None for n, v in store.items()}

print("CalibrationSession OK")

In [ ]:
class QuantMethodComparator:
    # Why: prove GPTQ beats RTN on output error (Step 15 deep-dive)
    def __init__(self, cal: CalibrationSession):
        self.cal = cal

    def rtn_mse(self, layer, inputs, bits=4):
        # Why: baseline — naive round-to-nearest with no calibration
        qm = QuantLinearFactory.build(layer, RTNQuantizer(layer, bits).quantize()).to(layer.weight.device)
        return LinearMetrics.output_mse(layer, qm, inputs)

    def gptq_mse(self, layer, inputs, bits=4):
        # Why: GPTQ uses calibration Hessian — should win on output MSE
        gq = GPTQQuantizer(layer, bits)
        h = layer.register_forward_hook(lambda m, i, o, q=gq: q.add_batch(i[0].detach()))
        self.cal.run(2)
        h.remove()
        qm = QuantLinearFactory.build(layer, gq.quantize()).to(layer.weight.device)
        return LinearMetrics.output_mse(layer, qm, inputs)

# backward-compat aliases — Why: shorter names for notebook run cells
def build_quantized_linear(layer, state): return QuantLinearFactory.build(layer, state)
build_quantized_module = build_quantized_linear
def iter_linear_layers(root, limit=None): return LinearLayerScanner(root).layers(limit)
iter_linear_modules = iter_quantizable_modules = iter_linear_layers
def layer_num_params(l): return LinearMetrics.num_params(l)
def layer_weight_mse(a, b): return LinearMetrics.weight_mse(a, b)
def layer_output_mse(a, b, c): return LinearMetrics.output_mse(a, b, c)
def replace_module(m, n, mod): ModelPatcher.replace(m, n, mod)
GenericRTNQuantizer = RTNQuantizer

print("QuantMethodComparator + aliases OK")

## Step 2 — Load model

| Cell | Class | Output |
|------|-------|--------|
| 2a | `ImagePadder`, `FlorenceModelLoader` | loader helpers |
| 2b | `FlorenceOCRDetector` | OCR on image → lines + bboxes |
| 2c | run load | `processor`, `model`, `image` |

Phase A uses `load_fresh_model()` copy — main `model` stays fp16 until Phase D.


In [ ]:
# 2a — Loader classes
ensure_transformers()
from transformers import AutoProcessor, AutoModelForCausalLM

class ImagePadder:
    # Why: Florence-2 needs square input — pad document to square canvas
    @staticmethod
    def pad(image):
        w, h = image.size; side = max(w, h)
        c = Image.new("RGB", (side, side), "white")
        px, py = (side-w)//2, (side-h)//2
        c.paste(image, (px, py))
        return c, px, py, w, h

class FlorenceModelLoader:
    # Why: centralize model download + dtype selection (fp16 on GPU)
    def __init__(self, model_id=MODEL_ID, device=DEVICE):
        self.model_id, self.device = model_id, device
        self.dtype = torch.float16 if device == "cuda" else torch.float32

    def load_processor(self):
        # Why: Florence-2 has custom tokenizer + image preprocessor
        return AutoProcessor.from_pretrained(self.model_id, trust_remote_code=True)

    def load_model(self):
        # Why: load fp16 Florence-2 — main model stays fp16 until Phase D
        m = AutoModelForCausalLM.from_pretrained(
            self.model_id, trust_remote_code=True,
            torch_dtype=self.dtype, attn_implementation="eager").to(self.device)
        m.eval(); return m

    def fresh_copy(self):
        # Why: Phase A needs a throwaway copy so main model stays fp16
        return self.load_model()

print("ImagePadder + FlorenceModelLoader OK")


In [ ]:
class FlorenceOCRDetector:
    # Why: every phase measures quality by OCR line count, not just weight MSE
    def __init__(self, processor, prompt=OCR_PROMPT):
        self.processor, self.prompt = processor, prompt

    def detect(self, image, model, device, max_new_tokens=512):
        # Why: run Florence OCR and return text lines + bounding boxes
        padded, px, py, ow, oh = ImagePadder.pad(image)
        inp = self.processor(text=self.prompt, images=padded, return_tensors="pt").to(device)
        inp["pixel_values"] = inp["pixel_values"].to(dtype=next(model.parameters()).dtype)
        t0 = time.time()
        with torch.no_grad():
            gen = model.generate(input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
                                 max_new_tokens=max_new_tokens, num_beams=1, use_cache=False)
        elapsed = time.time() - t0
        raw = self.processor.batch_decode(gen, skip_special_tokens=False)[0]
        parsed = self.processor.post_process_generation(raw, task=self.prompt, image_size=(max(ow,oh),)*2)
        region = parsed.get(self.prompt, {})
        lines = []
        for quad, label in zip(region.get("quad_boxes",[]), region.get("labels",[])):
            xs, ys = quad[0::2], quad[1::2]
            b = [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]
            x1,y1 = max(0,min(ow,b[0]-px)), max(0,min(oh,b[1]-py))
            x2,y2 = max(0,min(ow,b[2]-px)), max(0,min(oh,b[3]-py))
            if x2>x1 and y2>y1:
                lines.append({"text": re.sub(r"</?\\w+[^>]*>","",str(label)).strip(),
                              "bbox": [x1,y1,x2,y2]})
        return lines, elapsed

def pad_info(img):
    # Why: expose padding offsets for bbox coordinate correction
    return ImagePadder.pad(img)

def run_florence_detect(img, proc, m, dev, max_new_tokens=512):
    # Why: one-liner wrapper for quick OCR calls in run cells
    return FlorenceOCRDetector(proc).detect(img, m, dev, max_new_tokens)

print("FlorenceOCRDetector OK")

In [ ]:
_loader = FlorenceModelLoader()
processor = _loader.load_processor()
model = _loader.load_model()
dtype = _loader.dtype

def load_fresh_model():
    # Why: get a new fp16 copy for Phase A without touching main model
    return _loader.fresh_copy()

try:
    url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
    image = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
except Exception:
    # Why: fallback so notebook still runs if download fails
    image = Image.new("RGB", (640,480), "white")
    ImageDraw.Draw(image).text((20,20), "Sample", fill="black")

print(f"Loaded {MODEL_ID} — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
print(f"Calibration image: {image.size[0]}×{image.size[1]} px")
plt.figure(figsize=(6,4)); plt.imshow(image); plt.title("Calibration page"); plt.axis("off"); plt.show()

## Phase A — Naive baseline · `NaiveQuantBaseline`

**Q:** What if every `nn.Linear` → int4 blindly? **Why:** worst-case floor for scorecard.

Run: define class → quantize `model_naive` → OCR compare → saves `baseline_ocr`.


In [ ]:
@dataclass
class NaiveProfile:
    # Why: one row per layer — tracks compression and weight error for Phase A
    name: str; bits: int; weight_mse: float; fp_bytes: int; q_bytes: int

LayerProfile = NaiveProfile

class NaiveQuantBaseline:
    # Why: Phase A — int4 everything blindly to set worst-case floor
    def __init__(self, n_bits=4, device=DEVICE):
        self.n_bits, self.device = n_bits, device

    def run(self, model, max_layers=MAX_QUANT_LAYERS) -> list[NaiveProfile]:
        # Why: RTN int4 every nn.Linear and swap in-place on model_naive
        profiles = []
        layers = LinearLayerScanner(model).layers(max_layers)
        for i, (name, layer) in enumerate(layers, 1):
            ql = QuantLinearFactory.build(layer, RTNQuantizer(layer, self.n_bits).quantize()).to(self.device)
            profiles.append(NaiveProfile(name, self.n_bits, LinearMetrics.weight_mse(layer, ql),
                LinearMetrics.num_params(layer)*2, ql.storage_bytes()))
            ModelPatcher.replace(model, name, ql)
            if i % 20 == 0 or i == len(layers): print(f"  {i}/{len(layers)} layers")
        return profiles

def apply_naive_uniform_quant(model, n_bits=4):
    # Why: shorthand alias for Phase A run cell
    return NaiveQuantBaseline(n_bits).run(model)

if __name__ == "__main__":
    tiny = nn.Sequential(nn.Linear(4,2), nn.Linear(2,1)).to(DEVICE)
    print(f"Demo: {len(NaiveQuantBaseline(4).run(tiny))} NaiveProfile rows")


In [ ]:
# Stage 8b — run naive baseline
print("Loading fp16 copy → model_naive")
model_naive = load_fresh_model()
t0 = time.time()
profiles_a = NaiveQuantBaseline(n_bits=4).run(model_naive)
elapsed = time.time() - t0
if profiles_a:
    avg = sum(p.weight_mse for p in profiles_a) / len(profiles_a)
    comp = sum(p.fp_bytes for p in profiles_a) / max(sum(p.q_bytes for p in profiles_a), 1)
    print(f"Done {elapsed:.1f}s | {len(profiles_a)} layers | avg MSE {avg:.2e} | {comp:.1f}× compression")


In [ ]:
# Stage 8c — OCR baseline
detector = FlorenceOCRDetector(processor)
ref = load_fresh_model()
fp16_lines, fp16_t = detector.detect(image, ref, DEVICE)
del ref
if DEVICE == "cuda": torch.cuda.empty_cache()
naive_lines, naive_t = detector.detect(image, model_naive, DEVICE)
print(f"{'':12} {'Lines':>6} {'Time':>8}")
print(f"{'fp16':12} {len(fp16_lines):>6} {fp16_t:>8.2f}s")
print(f"{'naive int4':12} {len(naive_lines):>6} {naive_t:>8.2f}s")
print(f"Δ lines: {len(naive_lines)-len(fp16_lines):+d}")
baseline_ocr = {"fp16_lines": len(fp16_lines), "naive_lines": len(naive_lines),
                "fp16_infer_s": fp16_t, "naive_infer_s": naive_t}


## Phase B — Scan · `LinearInventory`

**Q:** Which layers are biggest / protected? **Why:** can't plan bits without inventory.

Run: define class → print top layers + chart → `profiles_b`.


In [ ]:
@dataclass
class InventoryProfile:
    # Why: one row per nn.Linear — size, protection status for Phase B
    name: str; module_type: str; shape: tuple; num_params: int
    pct_of_linear: float; protected: bool; tiny: bool

class LinearInventory:
    # Why: Phase B — find where model weights live before assigning bits
    def __init__(self, model):
        self.model = model
        self.scanner = LinearLayerScanner(model)

    def summary(self):
        # Why: quick stats — how many Linear layers, what % of total params
        layers = self.scanner.layers()
        lp = sum(LinearMetrics.num_params(m) for _, m in layers)
        mp = sum(p.numel() for p in self.model.parameters())
        return {"count": len(layers), "linear_params": lp, "model_params": mp,
                "pct": 100 * lp / max(mp, 1)}

    def build(self, limit=None):
        # Why: full ranked list of layers for inventory table + chart
        layers = self.scanner.layers(limit)
        total = sum(LinearMetrics.num_params(m) for _, m in layers)
        rows = [InventoryProfile(n, "Linear", tuple(m.weight.shape), LinearMetrics.num_params(m),
            100*LinearMetrics.num_params(m)/max(total,1), self.scanner.is_protected(n),
            LinearMetrics.num_params(m) < MIN_PARAMS_TO_QUANT) for n, m in layers]
        rows.sort(key=lambda p: p.num_params, reverse=True)
        return rows, total

def build_layer_inventory(layers):
    # Why: build inventory from pre-scanned layer list (used in run cell)
    total = sum(LinearMetrics.num_params(m) for _, m in layers)
    sc = LinearLayerScanner(nn.Linear(1,1))
    rows = [InventoryProfile(n,"Linear",tuple(m.weight.shape),LinearMetrics.num_params(m),
        100*LinearMetrics.num_params(m)/max(total,1), sc.is_protected(n),
        LinearMetrics.num_params(m)<MIN_PARAMS_TO_QUANT) for n,m in layers]
    rows.sort(key=lambda p: p.num_params, reverse=True)
    return rows, total

def linear_layer_summary(model):
    # Why: one-liner for quick layer count / param share
    inv = LinearInventory(model)
    return inv.summary()

LayerProfile = InventoryProfile

if __name__ == "__main__":
  inv = LinearInventory(model)
  print("summary:", inv.summary())


In [ ]:
# Stage 9 — inventory
inv = LinearInventory(model)
summary = inv.summary()
layers = LinearLayerScanner(model).layers(MAX_ANALYZE_LAYERS or MAX_QUANT_LAYERS)
profiles_b, _ = build_layer_inventory(layers)
print(f"Linear layers: {summary['count']} ({summary['pct']:.1f}% of model params)\n")
print(f"{'Rank':<5} {'Params':>10}  Name")
for i, r in enumerate(profiles_b[:12], 1):
    tag = " [PROTECT]" if r.protected else ""
    print(f"{i:<5} {r.num_params:>10,}  {r.name.split('.')[-1]}{tag}")
fig, ax = plt.subplots(figsize=(10,4))
show = profiles_b[:12]
ax.barh([p.name.split(".")[-1] for p in show][::-1], [p.pct_of_linear for p in show][::-1])
ax.set_xlabel("% of Linear params"); ax.set_title("Biggest nn.Linear layers")
plt.tight_layout(); plt.show()


## Phase C — Hook + Rank · `SensitivityAnalyzer`

**Q:** Which layer breaks OCR when quantized? **Why:** weight MSE alone misleads.

Run: define class → `collect_inputs()` (hooks) → rank → `profiles_c` + `captures`.


In [ ]:
@dataclass
class SensitivityProfile:
    # Why: one row per layer — output MSE + sensitivity rank for Phase C
    name: str; module_type: str; shape: tuple; num_params: int; protected: bool
    weight_mse_int4: float; weight_mse_int8: float
    output_mse_int4: float; output_mse_int8: float; act_max: float; sensitivity: float

class SensitivityAnalyzer:
    # Why: Phase C — rank layers by how much quantizing them hurts OCR output
    def __init__(self, calibrator: CalibrationSession):
        self.cal = calibrator

    def _w_mse(self, layer, bits):
        # Why: weight-only error (cheap but incomplete — used for reference)
        sq = SymmetricQuantizer(bits)
        q, s = sq.quantize(layer.weight.float())
        return (layer.weight.float() - sq.dequantize(q,s)).pow(2).mean().item()

    def _o_mse(self, layer, inputs, bits):
        # Why: output error on real OCR activations — the metric that matters
        if inputs is None or inputs.numel()==0: return float("nan")
        qm = QuantLinearFactory.build(layer, RTNQuantizer(layer,bits).quantize()).to(layer.weight.device)
        return LinearMetrics.output_mse(layer, qm, inputs)

    def analyze(self, profiles_b, layers, captures) -> list[SensitivityProfile]:
        # Why: rank all layers — high sensitivity → keep fp16/int8 in Phase D
        inv = {p.name: p for p in profiles_b}
        rows = []
        for name, layer in layers:
            p = inv[name]; inp = captures.get(name)
            am = float(inp.abs().max()) if inp is not None else 0.
            o4, o8 = self._o_mse(layer, inp, 4), self._o_mse(layer, inp, 8)
            sens = o4 * (1 + 0.1*math.log1p(am)) if not math.isnan(o4) else 0.
            rows.append(SensitivityProfile(name,"Linear",tuple(layer.weight.shape),p.num_params,p.protected,
                self._w_mse(layer,4), self._w_mse(layer,8), o4, o8, am, sens))
        rows.sort(key=lambda r: r.sensitivity, reverse=True)
        return rows

def build_quantizer(layer, n_bits=None):
    # Why: create GPTQ quantizer with config defaults
    return GPTQQuantizer(layer, n_bits or BITS, GPTQ_BLOCK_SIZE, GPTQ_DAMPING)
def build_quantizer_for_module(layer, n_bits=None): return build_quantizer(layer, n_bits)
def analyze_sensitivity(pb, layers, cap):
    return SensitivityAnalyzer(CalibrationSession(model,processor,image,OCR_PROMPT)).analyze(pb,layers,cap)
def run_calibration(m,p,img,prompt,n):
    return CalibrationSession(m,p,img,prompt).run(n)
def register_calibration_hooks(m,q):
    return CalibrationSession(m,processor,image,OCR_PROMPT).register_gptq_hooks(q)
def collect_layer_inputs(m,names,p,img,prompt,nb):
    return CalibrationSession(m,p,img,prompt).collect_inputs(names,nb)

LayerProfile = SensitivityProfile

if __name__ == "__main__":
    print("SensitivityAnalyzer ready — run Stage 10 cell")


In [ ]:
# Stage 10 — sensitivity
cal = CalibrationSession(model, processor, image, OCR_PROMPT)
layers = LinearLayerScanner(model).layers(MAX_ANALYZE_LAYERS or MAX_QUANT_LAYERS)
names = [n for n,_ in layers]
print(f"Capturing inputs for {len(names)} layers...")
captures = cal.collect_inputs(names, MAX_CALIB_BATCHES)
profiles_c = SensitivityAnalyzer(cal).analyze(profiles_b, layers, captures)
print(f"\n{'Rank':<5} {'Sensitivity':>11}  Layer")
for i,p in enumerate(profiles_c[:12],1):
    print(f"{i:<5} {p.sensitivity:>11.2e}  {p.name.split('.')[-1]}{' [PROTECT]' if p.protected else ''}")
fig, ax = plt.subplots(1,2, figsize=(14,5))
show = profiles_c[:15]
lbl = [p.name.split(".")[-1] for p in show]
ax[0].barh(lbl[::-1], [p.sensitivity for p in show][::-1], color="#e74c3c")
ax[0].set_title("Sensitivity (higher = more fragile)")
ax[1].barh(lbl[::-1], [p.output_mse_int4 for p in show][::-1], color="#3498db")
ax[1].set_title("Output MSE @ int4")
plt.tight_layout(); plt.show()


## Phase D — GPTQ · `QuantPlanBuilder` + `GPTQApplier`

**Q:** Can GPTQ beat Phase A at similar size? **Why:** mixed int4/int8/fp16 + Hessian quant.

Run: define plan class → define applier → build plan → apply GPTQ (hooks per layer) → chart → OCR → scorecard.


In [ ]:
# Phase D — QuantPlanBuilder
@dataclass
class PlanProfile:
    # Why: one row per layer — assigned bits + error metrics for Phase D
    name: str; module_type: str; num_params: int; sensitivity: float; protected: bool
    bits: str = "int4"; note: str = ""
    weight_mse: float = 0.; output_mse: float = 0.
    fp_bytes: int = 0; q_bytes: int = 0; applied: bool = False

class QuantPlanBuilder:
    # Why: assign int4/int8/fp16 per layer based on sensitivity rank
    def __init__(self, fp16_pct=FP16_SENSITIVE_PCT, int8_pct=INT8_MID_PCT):
        self.fp16_pct, self.int8_pct = fp16_pct, int8_pct

    def build_mixed(self, profiles_c) -> list[PlanProfile]:
        # Why: top-sensitive → fp16, medium → int8, rest → int4
        plan = [PlanProfile(r.name,"Linear",r.num_params,r.sensitivity,r.protected) for r in profiles_c]
        for e in plan:
            if e.protected: e.bits, e.note = "fp16", "protected"
        cand = [e for e in plan if not e.protected]
        n = len(cand)
        if not n: return plan
        n16 = max(1, round(n*self.fp16_pct/100))
        n8 = max(0, round(n*self.int8_pct/100))
        for i,e in enumerate(cand):
            if i<n16: e.bits,e.note = "fp16", f"high sens rank {i+1}"
            elif i<n16+n8: e.bits,e.note = "int8", "medium sens"
            else: e.bits,e.note = "int4", "low sens"
        return plan

    def build_uniform(self, profiles_c, bits):
        # Why: fallback when MIXED_PRECISION=False — all layers same bit width
        return [PlanProfile(r.name,"Linear",r.num_params,r.sensitivity,r.protected,
            "fp16" if r.protected else f"int{bits}", "protected" if r.protected else f"uniform int{bits}")
            for r in profiles_c]

def build_quant_plan(pc):
    # Why: shorthand for mixed-precision plan
    return QuantPlanBuilder().build_mixed(pc)
def build_uniform_plan(pc,b):
    return QuantPlanBuilder().build_uniform(pc,b)
LayerProfile = PlanProfile
print("QuantPlanBuilder OK")


In [ ]:
class GPTQApplier:
    # Why: Phase D — the only class that modifies the main model
    def __init__(self, calibrator: CalibrationSession, device=DEVICE):
        self.cal, self.device = calibrator, device

    def apply(self, model, plan, captures=None) -> list[PlanProfile]:
        # Why: for each int4/int8 layer — calibrate GPTQ, swap to GPTQLinear
        todo = [p for p in plan if p.bits in ("int4","int8")]
        if not todo: print("Nothing to quantize."); return []
        layers = dict(LinearLayerScanner(model).layers())
        applied = []
        print(f"\nGPTQApplier: {len(todo)} layers\n")
        for i, entry in enumerate(todo, 1):
            if entry.name not in layers or not isinstance(layers[entry.name], nn.Linear): continue
            layer = layers[entry.name]
            nb = int(entry.bits.replace("int",""))
            print(f"  [{i}/{len(todo)}] {entry.name}  bits={entry.bits}")
            gq = GPTQQuantizer(layer, nb, GPTQ_BLOCK_SIZE, GPTQ_DAMPING)
            hs = self.cal.register_gptq_hooks({entry.name: gq})
            self.cal.run(MAX_CALIB_BATCHES)
            for h in hs: h.remove()
            ql = QuantLinearFactory.build(layer, gq.quantize()).to(self.device)
            entry.weight_mse = LinearMetrics.weight_mse(layer, ql)
            entry.output_mse = LinearMetrics.output_mse(layer, ql, captures.get(entry.name) if captures else None)
            entry.fp_bytes = LinearMetrics.num_params(layer)*2
            entry.q_bytes = ql.storage_bytes(); entry.applied = True
            ModelPatcher.replace(model, entry.name, ql)
            applied.append(entry)
            print(f"       → GPTQLinear  mse={entry.weight_mse:.2e}")
        return applied

def apply_quant_plan(m, pd, cap=None):
    # Why: one-liner to run GPTQ on all planned layers
    return GPTQApplier(CalibrationSession(m,processor,image,OCR_PROMPT)).apply(m,pd,cap)

LayerProfile = PlanProfile
print("GPTQApplier OK")

In [ ]:
# Stage 11 — build plan
builder = QuantPlanBuilder()
profiles_d = builder.build_mixed(profiles_c) if MIXED_PRECISION else builder.build_uniform(profiles_c, BITS)
from collections import Counter
bc = Counter(p.bits for p in profiles_d)
print("Bit plan:", dict(bc))
for p in profiles_d[:15]:
    print(f"  {p.bits:5} {p.name.split('.')[-1]:20}  {p.note}")
fig,ax=plt.subplots(figsize=(5,4))
ax.pie([sum(p.num_params for p in profiles_d if p.bits==b) for b in ("int4","int8","fp16")],
       labels=["int4","int8","fp16"], autopct="%1.0f%%"); ax.set_title("Param share by precision")
plt.show()
total_quant_params = sum(p.num_params for p in profiles_d)


In [ ]:
# Stage 12 — apply GPTQ
cal = CalibrationSession(model, processor, image, OCR_PROMPT)
applier = GPTQApplier(cal)
n_quant = sum(1 for p in profiles_d if p.bits in ("int4","int8"))
n_fp16 = sum(1 for p in profiles_d if p.bits == "fp16")
print(f"Quantizing {n_quant} layers, keeping {n_fp16} in fp16")
profiles_d_applied = applier.apply(model, profiles_d, captures)
print(f"\nSwapped {len(profiles_d_applied)} layers to GPTQLinear")


In [ ]:
# Stage 12b — weight error chart
if profiles_d_applied:
    fig, ax = plt.subplots(figsize=(10,4))
    names = [p.name.split(".")[-1] for p in profiles_d_applied]
    colors = {"int4":"#27ae60","int8":"#f39c12"}
    ax.barh(names[::-1], [p.weight_mse for p in profiles_d_applied][::-1],
            color=[colors.get(p.bits,"gray") for p in profiles_d_applied][::-1])
    ax.set_xlabel("Weight MSE"); ax.set_title("GPTQ weight error per layer")
    plt.tight_layout(); plt.show()


### Step 13 — OCR on GPTQ `model` (not `model_naive`)


In [ ]:
detector = FlorenceOCRDetector(processor)
lines, infer_s = detector.detect(image, model, DEVICE)
print(f"GPTQ model: {len(lines)} lines in {infer_s:.2f}s")
for line in lines[:6]:
    print(f"  • {line['text'][:65]}")
vis = image.copy(); draw = ImageDraw.Draw(vis)
for line in lines:
    b = line["bbox"]; draw.rectangle(b, outline="lime", width=2)
plt.figure(figsize=(8,6)); plt.imshow(vis)
plt.title(f"GPTQ detect — {len(lines)} lines"); plt.axis("off"); plt.show()


### Step 14 — Scorecard: fp16 vs naive int4 vs GPTQ mixed


In [ ]:
gptq_lines = len(lines)
ref = baseline_ocr
print(f"{'Approach':<14} {'Lines':>6} {'Δ fp16':>8} {'Time':>8}")
print("-"*40)
print(f"{'fp16':<14} {ref['fp16_lines']:>6} {'—':>8} {ref['fp16_infer_s']:>7.2f}s")
print(f"{'naive int4':<14} {ref['naive_lines']:>6} {ref['naive_lines']-ref['fp16_lines']:>+8d} {ref['naive_infer_s']:>7.2f}s")
print(f"{'GPTQ mixed':<14} {gptq_lines:>6} {gptq_lines-ref['fp16_lines']:>+8d} {infer_s:>7.2f}s")
fig, ax = plt.subplots(figsize=(5,3))
ax.bar(["fp16","naive int4","GPTQ"], [ref["fp16_lines"],ref["naive_lines"],gptq_lines],
       color=["#3498db","#e74c3c","#27ae60"])
ax.set_ylabel("Detected lines"); ax.set_title("Phase A naive vs Phase D GPTQ")
plt.tight_layout(); plt.show()
if gptq_lines >= ref["naive_lines"]:
    print("\n✓ GPTQ matched or beat naive baseline on line count.")


### Step 15 (optional) — `QuantMethodComparator`: RTN vs GPTQ on one layer


In [ ]:
# Stage 15 — GPTQ vs RTN on one layer
cands = [(n, m) for n, m in LinearLayerScanner(model).layers(8)
         if m.weight.numel() >= MIN_PARAMS_TO_QUANT]
if cands and captures:
    dn, dl = cands[0]
    xin = captures.get(dn)
    if xin is not None and xin.numel():
        cmp = QuantMethodComparator(CalibrationSession(model, processor, image, OCR_PROMPT))
        mr, mg = cmp.rtn_mse(dl, xin), cmp.gptq_mse(dl, xin)
        print(f"Layer: {dn}")
        print(f"  RTN  output MSE: {mr:.4e}")
        print(f"  GPTQ output MSE: {mg:.4e}")
        print(f"  GPTQ wins by {(1 - mg / max(mr, 1e-12)) * 100:.1f}%")
else:
    print("Run Phase C first — need captures from sensitivity step.")


## Recap

```
A: NaiveQuantBaseline → baseline_ocr
B: LinearInventory → profiles_b
C: CalibrationSession + SensitivityAnalyzer → profiles_c, captures
D: QuantPlanBuilder + GPTQApplier → GPTQ model → beat baseline ✓
```

**Hooks:** required for GPTQ, AWQ, SmoothQuant, sensitivity — **not** for RTN.

**Next:** [03 — Mobile export](03_ocr_pipeline_mobile.ipynb)
